In [1]:
import os
import pandas as pd
import s3fs
import sklearn

In [2]:
sklearn.__version__

'1.7.0'

In [3]:
os.environ['AWS_S3_ENDPOINT']

'minio-simple.lab.groupe-genes.fr'

In [4]:
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL

'http://minio-simple.lab.groupe-genes.fr'

In [5]:
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })

In [6]:
fs.ls('')

['ematzner-ensae']

In [7]:
fs.ls('ematzner-ensae')

['ematzner-ensae/Mllib',
 'ematzner-ensae/Prix20181114.csv',
 'ematzner-ensae/Recommendation',
 'ematzner-ensae/Streaming',
 'ematzner-ensae/dataini.parquet',
 'ematzner-ensae/dataset.parquet',
 'ematzner-ensae/flight-data',
 'ematzner-ensae/readmission_avc.parquet',
 'ematzner-ensae/retail-org',
 'ematzner-ensae/top_clients_by_category']

In [8]:
BUCKET = 'ematzner-ensae'
FILE_KEY_S3 = '/readmission_avc.parquet'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3

In [9]:
FILE_PATH_S3

'ematzner-ensae/readmission_avc.parquet'

In [10]:
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    dataini = pd.read_parquet(file_in)

In [11]:
dataini.dtypes

modeEntree      int32
modeSortie      int32
duree           int32
ghm2           object
dp             object
sexe          float64
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
dtype: object

# 1. Data Preprocessing

In [12]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           20
age            20
nbActe          0
nbRum           0
nbda          134
id              0
id_D          200
dtype: int64

In [13]:
dataini = dataini.dropna(subset=['id_D'], axis= 0).copy()

In [14]:
dataini["rea"] = (dataini["id_D"] != "").astype("int8") # int8 → 1 octet par valeur
# convertir en boolean

In [15]:
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


In [16]:
dataini.rea.value_counts()

rea
0    1281
1     219
Name: count, dtype: int64

In [17]:
dataini.dtypes

modeEntree      int32
modeSortie      int32
duree           int32
ghm2           object
dp             object
sexe          float64
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
rea              int8
dtype: object

In [18]:
str_cols = ['modeEntree', 'modeSortie', 'sexe']


In [19]:
# contre exemple
# dataini[str_cols] = dataini[str_cols].astype('str')

In [20]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           19
age            19
nbActe          0
nbRum           0
nbda          121
id              0
id_D            0
rea             0
dtype: int64

In [21]:
dataini[str_cols] = dataini[str_cols].astype("object")

In [22]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           19
age            19
nbActe          0
nbRum           0
nbda          121
id              0
id_D            0
rea             0
dtype: int64

In [23]:
dataini.nbda.value_counts()

nbda
3.0     193
4.0     178
2.0     154
1.0     150
5.0     135
6.0     129
7.0     115
8.0      85
9.0      68
10.0     37
11.0     32
13.0     27
12.0     20
14.0     14
15.0     13
16.0      8
17.0      4
18.0      4
26.0      3
19.0      3
23.0      2
27.0      1
24.0      1
21.0      1
20.0      1
22.0      1
Name: count, dtype: int64

In [24]:
dataini['nbda'] = dataini['nbda'].fillna(0)

In [25]:
dataset = dataini[dataini['modeSortie'] != 9]

In [26]:
dataset = dataset.drop(['id', 'id_D'], axis = 1)

In [27]:
dataset.to_parquet('dataset.parquet')

In [28]:
# on suppose que fs est déjà créé (s3fs.S3FileSystem)
OUT_PATH_S3 = BUCKET+"/dataini.parquet"

with fs.open(OUT_PATH_S3, mode="wb") as file_out:
    dataini.to_parquet(file_out, engine="pyarrow", index=False, compression="snappy")


In [29]:
OUT_PATH_S3

'ematzner-ensae/dataini.parquet'

# 2. Feature Engineering
## 2.1 Principes

In [30]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer

# StandardScaler → agit par variable (colonne), met chaque feature à moyenne 0 et écart-type 1.

# Normalizer → agit sur chaque individu (ligne).
# "Norme 1" → le vecteur est réduit à une longueur 1, mais garde la même direction (il pointe dans le même sens).

In [31]:
dataset['sexe']

1       2.0
2       NaN
4       1.0
6       1.0
7       2.0
       ... 
1695    1.0
1696    1.0
1697    1.0
1698    2.0
1699    1.0
Name: sexe, Length: 1322, dtype: object

In [32]:
OneHotEncoder(sparse_output=False, drop='first').fit_transform(SimpleImputer(strategy= 'most_frequent').fit_transform(dataset[['sexe']]))

array([[1.],
       [0.],
       [0.],
       ...,
       [0.],
       [1.],
       [0.]], shape=(1322, 1))

## 2.2 Preprocessing Pipeline

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [34]:
features = dataset.drop('rea', axis = 1)
label = dataset['rea']

In [35]:
features.dtypes

modeEntree     object
modeSortie     object
duree           int32
ghm2           object
dp             object
sexe           object
age           float64
nbActe          int32
nbRum           int32
nbda          float64
dtype: object

In [36]:
num_features = features.select_dtypes(include= ['int32', 'float64']).columns
cat_features = features.select_dtypes(include= ['object']).columns

In [37]:
cat_features

Index(['modeEntree', 'modeSortie', 'ghm2', 'dp', 'sexe'], dtype='object')

In [38]:
# ou
# cat_features = list(set(features.columns) - set(num_features))

In [39]:
num_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

In [40]:
cat_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown= 'ignore'))
])

In [41]:
preprocessor = ColumnTransformer(transformers= [
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [42]:
features.columns

Index(['modeEntree', 'modeSortie', 'duree', 'ghm2', 'dp', 'sexe', 'age',
       'nbActe', 'nbRum', 'nbda'],
      dtype='object')

In [43]:
dataset.columns

Index(['modeEntree', 'modeSortie', 'duree', 'ghm2', 'dp', 'sexe', 'age',
       'nbActe', 'nbRum', 'nbda', 'rea'],
      dtype='object')

In [44]:
preprocessor.fit_transform(features).toarray()

array([[-0.68288188,  0.37261423, -0.29461677, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.35228593,  0.        , -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       [-0.16529798, -0.02521297, -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.16529798, -0.22412657, -0.23119333, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.14525237, -2.87630794,  0.46646456, ...,  0.        ,
         0.        ,  1.        ],
       [-0.78639866,  0.04109156, -0.54831055, ...,  0.        ,
         1.        ,  0.        ]], shape=(1322, 112))

In [45]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(features, label, test_size= 0.1, random_state= 18)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size= 0.1, random_state= 42)

In [46]:
features.shape

(1322, 10)

In [47]:
label.shape

(1322,)

In [48]:
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(1070, 10) (119, 10) (133, 10)
(1070,) (119,) (133,)


# 3. Model
## 3.1 Logistic Regression

In [49]:
from sklearn.linear_model import LogisticRegression

In [50]:
lr = LogisticRegression()

In [526]:
pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', lr)
])

In [421]:
pip_reg_fitted = pip_reg.fit(X_train, y_train)

In [422]:
y_predict = pip_reg_fitted.predict(X_val)

In [423]:
y_predict_proba = pip_reg_fitted.predict_proba(X_val)[:,1]

In [424]:
y_predict_proba

array([0.05434258, 0.97548684, 0.98286929, 0.0211812 , 0.0631269 ,
       0.10254158, 0.02537978, 0.08273014, 0.02228209, 0.12270215,
       0.09570909, 0.02193838, 0.11686945, 0.12456897, 0.09138028,
       0.03862216, 0.97576334, 0.97922886, 0.02889055, 0.097124  ,
       0.08275629, 0.9772158 , 0.07117104, 0.16967249, 0.01482169,
       0.08644865, 0.02315506, 0.01568778, 0.08471653, 0.02734098,
       0.08538556, 0.10187251, 0.07345369, 0.05570633, 0.08937385,
       0.02476259, 0.08447695, 0.02633935, 0.08653747, 0.07501422,
       0.11059345, 0.98219666, 0.05484911, 0.08380927, 0.03944364,
       0.09274702, 0.97169355, 0.09488123, 0.11668844, 0.04474209,
       0.09788602, 0.13422667, 0.03167464, 0.16393663, 0.06383311,
       0.03766595, 0.02792632, 0.03207453, 0.06229111, 0.01839435,
       0.10002437, 0.93301576, 0.02332209, 0.1178995 , 0.02983813,
       0.10429969, 0.12366031, 0.06291443, 0.02252857, 0.16511388,
       0.09183082, 0.04045147, 0.07448081, 0.07143891, 0.97775

In [425]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [426]:
accuracy_score(y_val, y_predict)

0.9495798319327731

In [427]:
f1_score(y_val, y_predict)

0.8235294117647058

In [428]:
roc_auc_score(y_val, y_predict_proba)

0.8207070707070707

## 3.2 RF

In [51]:
from sklearn.ensemble import RandomForestClassifier
from pprint import pprint

In [52]:
rf = RandomForestClassifier(random_state= 42)

In [431]:
pip_rf = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', rf)
]
)

In [432]:
pip_rf_fitted = pip_rf.fit(X_train, y_train)

In [433]:
rf_y_predict = pip_rf_fitted.predict(X_val)

In [434]:
rf_y_predict_proba = pip_rf_fitted.predict_proba(X_val)[:,1]

In [435]:
accuracy_score(y_val, rf_y_predict)

0.9495798319327731

In [436]:
# fitting 2 fois et montrer que le résultat est différent
roc_auc_score(y_val, rf_y_predict_proba)

0.8449494949494949

In [437]:
pprint(rf.get_params())

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}


**n_estimators** (Nombre d'arbres dans la forêt)<br/>

Pourquoi le tuner ?<br/>
* Augmenter le nombre d'arbres améliore souvent la performance du modèle (réduit la variance), mais au prix d'une augmentation du temps d'entraînement et d'inférence.
* Par défaut, 100 arbres suffisent souvent, mais augmenter à 200 ou 500 peut être utile pour des datasets complexes.
* Valeurs courantes : 50, 100, 200, 500.


**max_depth** (Profondeur maximale des arbres)<br/>
Description : Contrôle la profondeur maximale de chaque arbre.

Pourquoi le tuner ?<br/>
* Limiter la profondeur aide à éviter le surapprentissage (overfitting).
* Des arbres très profonds capturent trop de bruit dans les données, tandis que des arbres peu profonds risquent de sous-apprendre (underfitting).

Valeurs courantes :<br/>
* None (par défaut, arbre entièrement développé).
* Valeurs typiques : 5, 10, 20, en fonction de la taille des données.


**max_features** (Nombre de features à considérer lors de la division)<br/>
Description : Nombre de features à échantillonner aléatoirement pour rechercher la meilleure division à chaque split.<br/>
* 'sqrt' : 
* 'log2' : 
* Un entier : Un nombre fixe de features.<br/>

Pourquoi le tuner ?
* Moins de features à chaque split réduit la corrélation entre les arbres, ce qui améliore la diversité de la forêt.
* Trop peu de features peuvent limiter la capacité d'apprentissage des arbres.
* Valeurs courantes : 'sqrt', 'log2', 0.3, 0.5 (fraction des features).

**min_samples_leaf** (Nombre mininum d’échantillons par feuille)<br/>
Description : Nombre mininum d'échantillons requis pour qu'un nœud soit une feuille (Le nombre minimum d'individus (ou de lignes) requis pour qu’un nœud soit considéré comme une feuille.).<br/>
Pourquoi le tuner ?<br/>
* Augmenter cette valeur force les feuilles à contenir plus d’échantillons, ce qui réduit le surapprentissage.<br/>
* Cela est particulièrement utile pour les datasets déséquilibrés ou bruités.<br/>

Valeurs courantes : 1 (par défaut), 5, 10, 20.

In [53]:
param_rf = {
    'classifier__n_estimators' : [100, 200, 500],
    'classifier__max_depth' : [10, 20, None], #
    # 'max_features' : ['sqrt', 'log2']
}

# La profondeur max de ton arbre ne dépend pas du nombre de variables, mais du nombre d’échantillons, des critères de split et des paramètres (max_depth, min_samples_split, min_samples_leaf)

### Effet de l’hyperparamètre `max_features` avec 10 variables

- **`max_features='sqrt'`**  
  - On prend la racine carrée du nombre total de variables.  
  - Ici : sqrt(10) ≈ 3.16 → donc **3 variables** considérées à chaque split.  

- **`max_features='log2'`**  
  - On prend le logarithme en base 2 du nombre total de variables.  
  - Ici : log2(10) ≈ 3.32 → donc **3 variables** aussi.  

➡️ Avec 10 variables, `sqrt` et `log2` donnent **le même résultat**.


In [54]:
metric_grid = ['accuracy', 'f1', 'roc_auc']

In [55]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [77]:
# StratifiedKFold agit uniquement sur la répartition de la variable cible Y (les étiquettes de classes)

cv = StratifiedKFold(
    n_splits= 5,
    shuffle=True, # les données sont mélangées aléatoirement avant d’être découpées en folds.
    random_state=42, 
    
)


# En pratique : 
# Dataset grand et équilibré → StratifiedKFold simple suffit.
# Dataset petit ou instable → RepeatedStratifiedKFold est recommandé.

from sklearn.model_selection import RepeatedStratifiedKFold

# cv = RepeatedStratifiedKFold(
#     n_splits=5,   # nombre de folds
#     n_repeats=3,  # combien de fois on refait le CV avec des splits différents
#     random_state=42
# )

In [460]:
grid_rf = GridSearchCV(
    estimator= pip_rf,
    param_grid= param_rf,
    cv = cv,
    scoring = 'roc_auc', # metric selon lequel on détermine le best_result,
    refit = True # par defaut
)

In [461]:
grid_rf_fitted = grid_rf.fit(X_train_val, y_train_val)

In [444]:
grid_rf_fitted.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_classifier__max_depth', 'param_classifier__n_estimators', 'params', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score'])

In [445]:
grid_rf_fitted.best_params_

{'classifier__max_depth': None, 'classifier__n_estimators': 100}

In [446]:
grid_rf_fitted.best_score_

np.float64(0.8720970038380587)

In [447]:
grid_rf_fitted.score(X_test, y_test)

0.9019774011299435

In [448]:
# equivalent à
roc_auc_score(y_test, grid_rf_fitted.best_estimator_.predict_proba(X_test)[:,1])

0.9019774011299435

In [449]:
# equivalent à
# roc_auc_score(y_test, grid_rf_fitted.predict_proba(X_test)[:,1])

## 3.3 KNN

In [56]:
from sklearn.neighbors import KNeighborsClassifier

In [57]:
pprint(KNeighborsClassifier().get_params())

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}


#### Hyperparamètres courants à tuner pour KNN<br/>

**n_neighbors** (Nombre de voisins)<br/>
Description : Définit combien de voisins sont considérés pour prendre une décision.<br/>
Pourquoi le tuner ?<br/>
* Un nombre de voisins trop faible peut rendre le modèle sensible au bruit (overfitting).
* Un nombre trop élevé peut le rendre trop généralisant (underfitting).
<br/>Valeurs courantes : 3, 5, 7, 10, 15.

**weights** (Pondération des voisins)
Description : Définit si tous les voisins contribuent également ou si leur contribution est pondérée par la distance.
'uniform' : Tous les voisins ont le même poids.
'distance' : Les voisins proches ont plus de poids que les voisins éloignés.
Pourquoi le tuner ? <br/>
La pondération par distance peut améliorer la performance si les observations proches sont plus représentatives de la classe cible.
Valeurs courantes : 'uniform', 'distance'.


**metric** (Distance utilisée pour trouver les voisins)
Description : Définit la mesure de distance utilisée pour calculer la proximité entre les points.
* 'minkowski' : Distance généralisée.
* 'euclidean' : Distance euclidienne
* 'manhattan' : Distance Manhattan

Pourquoi le tuner ?<br/>
Certaines mesures de distance peuvent mieux capturer la structure des données en fonction du problème.
Valeurs courantes : 'euclidean', 'manhattan', 'minkowski'.

**p** (Paramètre pour la distance de Minkowski)
Description : Définit la puissance pour la distance de Minkowski.
* p=1 correspond à la distance Manhattan.
* p=2 correspond à la distance Euclidienne.
  
<br/>Pourquoi le tuner ?<br/>
* Ajuster la valeur de p permet d'explorer différents types de relations dans les données.
* Valeurs courantes : 1, 2, 3.


**algorithm** (Algorithme utilisé pour la recherche des voisins)
Description : Définit la méthode utilisée pour rechercher les voisins les plus proches.
- 'auto' : Choisit automatiquement l'algorithme le plus adapté.
- 'ball_tree' : Utilise une structure BallTree.
- 'kd_tree' : Utilise une structure KDTree.
- 'brute' : Recherche brute.
  
Pourquoi le tuner ?<br/>
Bien que 'auto' fonctionne dans la plupart des cas, certaines structures (comme kd_tree) peuvent être plus rapides pour des données spécifiques.



In [58]:
knn = KNeighborsClassifier()

In [456]:
pip_knn = Pipeline(steps=[
    ('preproc' , preprocessor),
    ('classifier' , knn)
    
])

In [59]:
param_knn = {
    'preproc__num__scaler' : [StandardScaler(), MinMaxScaler(), Normalizer()],
    'classifier__n_neighbors' : [5, 7, 10,],
    'classifier__weights' : ['uniform', 'distance'],
    'classifier__metric' : ['euclidean', 'manhattan']
}

In [476]:
grid_knn = GridSearchCV(
    estimator=pip_knn,
    param_grid=param_knn,
    cv = cv,
    scoring = 'roc_auc',
    refit = True
)

In [477]:
grid_knn_fitted = grid_knn.fit(X_train_val, y_train_val)

In [478]:
grid_knn_fitted.best_estimator_

,steps,"[('preproc', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [479]:
grid_knn_fitted.best_score_

np.float64(0.868425776897363)

In [480]:
grid_knn_fitted.best_params_

{'classifier__metric': 'manhattan',
 'classifier__n_neighbors': 10,
 'classifier__weights': 'distance',
 'preproc__num__scaler': Normalizer()}

In [481]:
grid_knn_fitted.score(X_test, y_test)

0.8740112994350282

In [482]:
roc_auc_score(y_test, grid_knn_fitted.best_estimator_.predict_proba(X_test)[:,1])

0.8740112994350282

## 3.4 Gradient Boosting

In [60]:
from sklearn.ensemble import GradientBoostingClassifier

In [61]:
gb = GradientBoostingClassifier()

In [62]:
pprint(gb.get_params())

{'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'log_loss',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': None,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}


**learning_rate** <br/>
Description : Le Gradient Boosting construit les arbres séquentiellement : chaque nouvel arbre est formé pour corriger les erreurs faites par les arbres précédents. À chaque étape, le nouvel arbre ajuste les résidus (erreurs) des prédictions précédentes.Le learning_rate contrôle la contribution de chaque nouvel arbre à la prédiction globale 𝐹 𝑡 ( 𝑥 )<br/>
Importance : Très élevé.<br/>
* Un taux d'apprentissage faible (ex. : 0.01) nécessite plus d'arbres (n_estimators) pour atteindre des performances optimales.
* Un taux élevé (ex. : 0.2 ou 0.3) permet une convergence rapide, mais risque de sur-apprendre.
* Valeur par défaut : 0.1.
* Valeurs courantes : [0.01, 0.05, 0.1, 0.2].


**n_estimators**<br/>
Description : Nombre total d'arbres dans le modèle. A chaque itération, un seul arbre est ajouté au modèle<br/>
Importance : Très élevé.
* Un nombre insuffisant d'arbres peut sous-apprendre, tandis qu'un nombre trop élevé peut augmenter le temps d'entraînement sans amélioration significative.
* Valeur par défaut : 100.
* Valeurs courantes : [100, 200, 500, 1000], dépendant de la taille des données et de learning_rate.

**max_depth**<br/>
Description : Profondeur maximale de chaque arbre.<br/>
Importance : Élevée.
* Limiter la profondeur empêche les arbres de surapprendre en capturant trop de bruit.
* Une profondeur trop faible peut sous-apprendre les relations complexes dans les données.
* Valeur par défaut : 3.
* Valeurs courantes : [3, 5, 7, None].

**min_samples_split**<br/>
Description : Nombre mininum d'échantillons requis pour diviser un nœud.<br/>
Importance : Moyen.<br/>
* Permet de contrôler la taille minimale des nœuds parents, ce qui limite les arbres très complexes.
* Valeur par défaut : 2.
* Valeurs courantes : [2, 5, 10].

**min_samples_leaf** <br/>
Description : Nombre mininum d'échantillons requis pour être dans une feuille.<br/>
Importance : Moyen.<br/>
* Empêche les arbres de créer des feuilles contenant un très petit nombre de données, ce qui peut les rendre sensibles au bruit.
* Valeur par défaut : 1.
* Valeurs courantes : [1, 5, 10, 20].

In [63]:
gb = GradientBoostingClassifier(random_state= 42)

In [489]:
pip_gb = Pipeline(steps = [
    ('preproc', preprocessor),
    ('classifier', gb)
])

In [64]:
param_gb = {
    'classifier__n_estimators' : [200, 500, ],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth' : [7, 10]
}

In [506]:
grid_gb = GridSearchCV(
    estimator=pip_gb,
    cv = cv,
    param_grid=param_gb,
    scoring = 'roc_auc'
)

In [507]:
grid_gb_fitted = grid_gb.fit(X_train_val, y_train_val)

In [496]:
grid_gb_fitted.best_estimator_

,steps,"[('preproc', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [508]:
grid_gb_fitted.best_score_

np.float64(0.8755986133465397)

In [509]:
grid_gb_fitted.best_params_

{'classifier__learning_rate': 0.05,
 'classifier__max_depth': 7,
 'classifier__n_estimators': 500}

In [510]:
grid_gb_fitted.score(X_test, y_test)

0.8977401129943503

In [512]:
roc_auc_score(y_test, grid_gb_fitted.best_estimator_.predict_proba(X_test)[:, 1])

0.8977401129943503

## 3.5 MLP

In [513]:
from sklearn.neural_network import MLPClassifier

In [515]:
mlp = MLPClassifier(random_state=42,
                   max_iter= 1000)

# 🔹 Synthèse MLPClassifier (scikit-learn)

### 1. Initialisation et aléatoire
- Les poids et biais du réseau sont initialisés aléatoirement.  
- `random_state` rend l’initialisation (et le shuffle des données) **reproductible**.  

---

### 2. Epochs, itérations et batchs
- **Mini-batch** : sous-ensemble du dataset utilisé pour une mise à jour des poids.  
- **Itération (step)** = 1 forward + 1 backward + 1 update sur **1 mini-batch**.  
- **Epoch** = passage complet sur tout le dataset → donc plusieurs itérations.  
- Relation :   nb_iterations = nb_epochs × nb_batchs_par_epoch


### 3. Paramètre `max_iter`
- Dans `MLPClassifier`, `max_iter` = **nombre maximum d’epochs**.  
- Ce n’est pas un hyperparamètre de tuning, mais une borne supérieure.  
- Exemple :  
- Dataset = 1000 exemples, `batch_size=100` → 10 batchs/epoch  
- `max_iter=200` → 200 epochs max → 200 × 10 = 2000 updates des poids.  

---

### 4. Convergence et arrêt
- Le modèle peut s’arrêter **avant `max_iter`** si la convergence est atteinte.  
- Critère d’arrêt :  
- `tol` : tolérance sur l’amélioration minimale,  
- `n_iter_no_change` : nb d’epochs consécutifs sans amélioration.  
- Avec `early_stopping=True`, l’arrêt se fait sur un jeu de validation.  

---

### 5. Hyperparamètres importants à tuner
- `hidden_layer_sizes` : architecture (nb de couches / nb de neurones).  
- `alpha` : régularisation L2.  
- `learning_rate_init` : taux d’apprentissage.  
- `activation` : fonction d’activation (`relu`, `tanh`, `logistic`).  
- `solver` : méthode d’optimisation (`adam`, `sgd`, `lbfgs`).  
- (Parfois `batch_size`).  

---

### 6. Résumé
- **Itération** = 1 batch → 1 update.  
- **Epoch** = tous les batchs → plusieurs updates.  
- **max_iter** = nb max d’epochs (donc plusieurs itérations).  
- L’entraînement peut s’arrêter avant convergence.


In [516]:
pip_mlp = Pipeline(
    steps=[
        ('preproc', preprocessor),
        ('classifier', mlp)
    ]
)

In [522]:
param_mlp = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam', 'sgd'], # l'algorithme d'optimisation utilisé pour mettre à jour les poids lors de l'entraînement du réseau de neurones
    'classifier__alpha': [0.0001, 0.001], # dans sklearn, régularisation est toujours ridge, alpha correspond à l'intensité de la pénalité
}

In [523]:
grid_mlp = GridSearchCV(
    estimator= pip_mlp,
    param_grid= param_mlp,
    cv = cv,
    scoring= 'roc_auc'
)

In [524]:
grid_mlp_fitted = grid_mlp.fit(X_train_val, y_train_val)

/opt/python/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perc

# 4. Model selection Pipeline 

In [65]:
models = [
    # ('lr', lr, {'classifier__C': [1.0]} ), # inverse de lambda, λ=1 → régularisation "standard"
    # ('rf', rf, param_rf),
    # ('knn', knn, param_knn),
    ('gb', gb, param_gb),
]

In [75]:
preproc_list = [
    # {'id' : 'basic', 'object': preprocessor},
    # {'id' : 'interact', 'object': preproc_interact},
    {'id' : 'spline', 'object': preproc_spline},
]

In [78]:
results = []
for preproc in preproc_list : 
    preproc_id = preproc['id']
    preproc_object = preproc['object']
    for model_id, model_object, param in models : 
        print(f'Model {model_id} with preprocessor {preproc_id}')
        pip = Pipeline(steps= [
            ('preproc', preproc_object),
            ('classifier', model_object)
        ])

        grid = GridSearchCV(
            estimator= pip,
            cv=cv, 
            param_grid = param,
            scoring= metric_grid,
            refit = 'roc_auc'
        ).fit(X_train_val, y_train_val)

        results.append(
            {
                'preprocessor' : preproc_id,
                'model' : model_id,
                'best_param' : grid.best_params_,
                'best_score' : grid.best_score_,
                'final_prediction' : grid.score(X_test, y_test)
            }
        )        

Model gb with preprocessor spline


In [79]:
pd.DataFrame(results)

,preprocessor,model,best_param,best_score,final_prediction
0,spline,gb,"{'classifier__learning_rate': 0.1, 'classifier...",0.868426,0.925989


# 5. More Feature Engineering
## 5.1 interaction

In [72]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

In [576]:
import numpy as np

In [577]:
X = np.arange(6).reshape(3,2)
X

array([[0, 1],
       [2, 3],
       [4, 5]])

In [581]:
poly = PolynomialFeatures(2, interaction_only=True)
poly.fit_transform(X)

array([[ 1.,  0.,  1.,  0.],
       [ 1.,  2.,  3.,  6.],
       [ 1.,  4.,  5., 20.]])

In [550]:
interact_transformer = Pipeline(steps=[
    ('imp', SimpleImputer()),
    ('poly', PolynomialFeatures(interaction_only=True)),
    ('scaler', StandardScaler())
])

In [69]:
fe_variables = ['age', 'duree', 'nbda']

In [70]:
no_fe_variable = num_features.drop(fe_variables)

In [574]:
preproc_interact = ColumnTransformer(transformers=[
    ('num', num_transformer, no_fe_variable),
    ('interact', interact_transformer, fe_variables),
    ('cat', cat_transformer, cat_features)
]
    
)

## 5.2 Spline

In [73]:
spline_transformer = Pipeline(steps=[
    ('imp', SimpleImputer()),
    ('poly', SplineTransformer()),
    ('scaler', StandardScaler())
])

In [74]:
preproc_spline = ColumnTransformer(transformers=[
    ('num', num_transformer, no_fe_variable),
    ('interact', spline_transformer, fe_variables),
    ('cat', cat_transformer, cat_features)
]   
)

## Pourquoi a-t-on besoin d’un jeu de test ?

### Rôles des jeux de données
- **Train set** : utilisé pour estimer les **paramètres** du modèle (poids, coefficients, splits…).  
- **Validation set (ou CV interne)** : utilisé pour choisir les **hyperparamètres** (ex. profondeur d’arbre, C en régression logistique, k en KNN).  
- **Test set** : totalement séparé, jamais vu → sert à mesurer la performance réelle du modèle final.

---

### Point important
- Les **hyperparamètres sont choisis en fonction du val set**.  
- Donc le **val set n’est pas neutre** : il influence directement le choix du modèle.
- les hyperparamètres marchent très bien sur la validation, mais généralisent mal → test set plus bas.

---

### Rôle du test set
- Fournir une **évaluation impartiale et neutre**.  
- Comme il n’a jamais servi ni à l’apprentissage, ni au choix des hyperparamètres, il reflète mieux la performance attendue en production.

---

### Résumé
- **Train** → apprend les paramètres.  
- **Val** → choisit les hyperparamètres (donc biaisé).  
- **Test** → seule vraie estimation neutre de la performance finale.


## Que faire si le score CV et le score Test sont différents ?

### Situation typique
- Score CV (moyenne des folds internes) = **0.85**  
- Score sur test = **0.75**  
Écart important → le modèle généralise mal.

---

### Causes possibles
| Cause | Signes | Solutions |
|-------|--------|-----------|
| **Sur-optimisation (overfitting sur le val set)** | CV très bon mais test beaucoup plus bas | Réduire l’espace d’hyperparamètres, simplifier le modèle, régulariser |
| **CV bruitée (instable)** | Scores CV très variables entre folds | Augmenter le nombre de folds (`cv=10`), utiliser `RepeatedStratifiedKFold`, vérifier la variance |
| **Test set différent (distribution shift)** | Test issu d’une autre source, période ou population | Vérifier la représentativité du split, collecter plus de données, envisager une adaptation de domaine |

---

### Règle d’or
- **Écart faible** (ex. CV=0.82, Test=0.80) → normal, cohérent.  
- **Écart fort** (ex. CV=0.85, Test=0.75) → problème de sur-tuning, de bruit, ou de distribution non représentative.  

Ne jamais utiliser le **test set** pour retuner les hyperparamètres → il doit rester **neutre** jusqu’à la fin.
